# How to Structure .claude/ for Maximum Efficiency

Run with Python 3.13 from the repository root. These tests use isolated fictional files and real formatter-hook processes, not a Claude model or a live account. They do not install settings into this repository or change global Claude configuration. Directory trees and Markdown workflows are documentation, not shell programs. Review optional example configuration before using it in your own project.

```bash
python -m pip install ruff==0.17.3 requests==2.34.2 pyyaml==6.0.3 jsonschema==4.26.0
```


In [ ]:
import os, json, sys, tempfile, shutil, subprocess, hashlib
from pathlib import Path
from importlib.metadata import version
import requests, yaml, jsonschema
project = Path.cwd()
assets = project / 'claude-project-structure'
assert (assets / 'format_edits.py').is_file()
fixture = Path(tempfile.mkdtemp(prefix='tdb-claude-structure-'))
for package in ['ruff','requests','pyyaml','jsonschema']:print(package,version(package))
print('Isolated fixture only; no Claude model calls or active settings changes.')

In [ ]:
"""A bounded PostToolUse formatter; not a general security sandbox."""
import json
import os
import subprocess
import sys
from pathlib import Path


def main():
    try:
        payload = sys.stdin.read(1_000_001)
        if len(payload) > 1_000_000:
            raise ValueError("Hook input exceeds the limit.")
        event = json.loads(payload)
        if not isinstance(event, dict):
            raise ValueError("Hook input must be a JSON object.")
        if event.get("hook_event_name") != "PostToolUse" or event.get("tool_name") not in {"Edit", "Write"}:
            return 0
        root = Path(os.environ["CLAUDE_PROJECT_DIR"]).resolve(strict=True)
        cwd = Path(event["cwd"]).resolve(strict=True)
        if cwd != root:
            raise ValueError("Hook cwd does not match the configured project.")
        name = event["tool_input"]["file_path"]
        if not isinstance(name, str) or not name or "\x00" in name:
            raise ValueError("Missing file path.")
        candidate = Path(name)
        target = (candidate if candidate.is_absolute() else root / candidate).resolve(strict=True)
        relative = target.relative_to(root)
        if any(part in {".git", ".venv", "node_modules"} for part in relative.parts):
            raise ValueError("Generated/internal paths are excluded.")
        if target.name.startswith(".env") or not target.is_file():
            raise ValueError("Expected an ordinary project source file.")
        if target.suffix != ".py":
            return 0
        result = subprocess.run([sys.executable, "-m", "ruff", "format", "--", str(target)],
            capture_output=True, text=True, timeout=10, check=False)
        if result.returncode:
            print(result.stderr, file=sys.stderr)
        return result.returncode
    except (ValueError, KeyError, TypeError, OSError, subprocess.TimeoutExpired) as error:
        print(f"Formatter hook: {error}", file=sys.stderr)
        return 2



print('Complete bounded formatter helper defined.')

In [ ]:
hook = (assets / 'format_edits.py').read_text()
wrapper = (assets / 'format-edits.sh').read_text()
hooks = fixture / '.claude/hooks'
hooks.mkdir(parents=True)
shutil.copy2(assets / 'format_edits.py',hooks / 'format_edits.py')
shutil.copy2(assets / 'format-edits.sh',hooks / 'format-edits.sh')
assert subprocess.run(['bash','-n',str(hooks / 'format-edits.sh')],capture_output=True).returncode == 0
compile(hook,'format_edits.py','exec')
print('Python helper compiles and Bash wrapper passes syntax validation.')

In [ ]:
settings = json.loads((assets / 'settings.example.json').read_text())
schema_url = settings['$schema']
response = requests.get(schema_url,timeout=30)
response.raise_for_status()
schema = response.json()
validator = jsonschema.validators.validator_for(schema)
validator.check_schema(schema)
validator(schema).validate(settings)
assert settings['hooks']['PostToolUse'][0]['matcher'] == 'Edit|Write'
assert settings['hooks']['PostToolUse'][0]['hooks'][0]['timeout'] == 15
print(json.dumps(settings,indent=2))
print('Settings pass the referenced JSON schema. SHA256:',hashlib.sha256(response.content).hexdigest())

In [ ]:
rule = (assets / 'backend-api.example.md').read_text()
parts = rule.split('---',2)
frontmatter = yaml.safe_load(parts[1])
assert frontmatter == {'paths':['app/api/**/*.py']}
assert '# Backend API Rules' in parts[2]
print(rule)
print('The paths field scopes this rule; a descriptive filename alone does not.')

In [ ]:
env = dict(os.environ, CLAUDE_PROJECT_DIR=str(fixture), PATH=str(Path(sys.executable).parent)+os.pathsep+os.environ.get('PATH',''))
def invoke(file_path, **overrides):
    event = {'hook_event_name':'PostToolUse','tool_name':'Edit','cwd':str(fixture),
             'tool_input':{'file_path':str(file_path)}}
    event.update(overrides)
    return subprocess.run(['bash',str(hooks / 'format-edits.sh')],input=json.dumps(event),
        text=True,capture_output=True,env=env,timeout=20)
for name in ['ordinary.py','with spaces.py','--looks-like-option.py','semi;colon.py']:
    path = fixture / name
    path.write_text('value={"answer":42}'+chr(10))
    result = invoke(path)
    assert result.returncode == 0,result.stderr
    assert path.read_text() == 'value = {"answer": 42}'+chr(10)
    assert invoke(path).returncode == 0
print('Actual Ruff formatting passes ordinary/space/leading-option/semicolon filenames and repeated execution.')

In [ ]:
outside = Path(tempfile.mkdtemp(prefix='tdb-claude-outside-'))
outside_file = outside / 'untouched.py'
outside_file.write_text('value={"answer":42}'+chr(10))
link = fixture / 'escape.py';link.symlink_to(outside_file)
secret = fixture / '.env.py';secret.write_text('fictional_fixture=1'+chr(10))
internal = fixture / '.git/internal.py';internal.parent.mkdir();internal.write_text('internal=1'+chr(10))
for path in [outside_file,link,secret,internal,fixture / 'missing.py']:
    assert invoke(path).returncode != 0
assert outside_file.read_text() == 'value={"answer":42}'+chr(10)
textfile = fixture / 'notes.txt';textfile.write_text('unchanged')
assert invoke(textfile).returncode == 0 and textfile.read_text() == 'unchanged'
assert invoke(fixture/'ordinary.py',cwd=str(outside)).returncode != 0
for malformed in ['{','null','[]']:
    result = subprocess.run(['bash',str(hooks / 'format-edits.sh')],input=malformed,
        text=True,capture_output=True,env=env,timeout=20)
    assert result.returncode == 2
print('Outside paths, symlink escape, secret/internal/missing files, cwd mismatch and malformed events reject.')
print('Non-Python documents remain unchanged. This hook is not a general security sandbox.')

In [ ]:
# Git checks run in the owned fixture, never the current repository.
subprocess.run(['git','init','--quiet',str(fixture)],check=True)
(fixture / '.gitignore').write_text('CLAUDE.local.md'+chr(10)+'.claude/settings.local.json'+chr(10))
(fixture / 'CLAUDE.md').write_text('# Shared project instructions'+chr(10))
(fixture / 'CLAUDE.local.md').write_text('# Fictional personal preferences'+chr(10))
(fixture / '.claude/settings.local.json').write_text('{}'+chr(10))
for name in ['CLAUDE.local.md','.claude/settings.local.json']:
    assert subprocess.run(['git','-C',str(fixture),'check-ignore','--quiet',name]).returncode == 0
assert subprocess.run(['git','-C',str(fixture),'check-ignore','--quiet','CLAUDE.md']).returncode == 1
print('Actual Git ignore checks separate shared instructions from both local preference files.')
print('No live Claude session, model quality, permission enforcement or security guarantee is claimed.')

## Original source examples and quotations

Nonexecuting complete archive material; the maintained formatter above replaces the original unsafe xargs pipeline.

```text
your-project/
├── CLAUDE.md                  # Main project instructions
├── CLAUDE.local.md            # Personal overrides (not committed)
└── .claude/
    ├── settings.json          # Control layer
    ├── rules/                 # Modular instructions
    ├── hooks/                 # Automation scripts
    ├── commands/              # Reusable prompt workflows
    ├── skills/                # Packaged capabilities
    └── agents/                # Specialized subagents
```

```text
your-project/
├── CLAUDE.md
└── .claude/
    └── rules/
        ├── frontend.md
        ├── backend-api.md
        ├── testing.md
        └── data-pipelines.md
```

```text
# Project: Customer Insights AP

## Stack
- FastAPI
- PostgreSQL
- SQLAlchemy
- Pytest

## Structure
- `app/api/` contains route definitions
- `app/services/` contains business logic
- `app/models/` contains ORM models
- `app/schemas/` contains request and response schemas

## Commands
- `pytest` runs the test suite
- `alembic upgrade head` applies migrations
- `ruff check .` runs linting
- `ruff format .` formats the code

## Conventions
- Validate all request bodies with Pydantic schemas
- Keep route handlers thin; business logic belongs in services
- Do not expose internal exception details in API responses
```

```text
# Backend API Rules

- Every new endpoint must include request and response schemas
- Use dependency injection for database sessions
- Return paginated results for collection endpoints
- Log external API failures with the shared logger
- Prefer service-layer functions over logic inside route files
```

```text
# Frontend Rules

- Prefer server components unless client interactivity is required
- Keep UI state local unless it is shared across pages
- Reuse design system components before creating new ones
- Put page-specific components beside their route when possible
```

```text
.claude/
└── rules/
    ├── frontend.md
    ├── backend-api.md
    ├── tests.md
    └── migrations.m
```

```text
.claude/
├── settings.json
├── hooks/
│   ├── block-dangerous-commands.sh
│   ├── format-edits.sh
│   └── run-tests-before-stop.sh
└── commands/
    ├── review-pr.md
    ├── write-tests.md
    └── summarize-changes.md
```

```text
#!/usr/bin/env bash
jq -r '.tool_input.file_path' | xargs ruff format
```

```text
.claude/
└── hooks/
    └── format-edits.sh
```

```text
.claude/
└── commands/
    ├── review-pr.md
    ├── fix-bug.md
    ├── write-tests.md
    └── prepare-release-notes.md
```

```text
# review-pr

Review the current changes with a focus on:
- correctness
- missing edge cases
- API contract changes
- test coverage gaps
Summarize:
1. critical issues
2. medium-risk issues
3. suggested improvements
```

```text
.claude/
└── commands/
    └── review-pr.md
```

```text
.claude/
├── settings.json
├── rules/
│   ├── frontend.md
│   ├── backend-api.md
│   └── testing.md
├── hooks/
│   ├── block-dangerous-commands.sh
│   ├── format-edits.sh
│   └── run-tests-before-stop.sh
└── commands/
    ├── review-pr.md
    ├── write-tests.md
    └── summarize-changes.md
```

```text
.claude/
├── hooks/
│   ├── lint-edits.sh
│   ├── block-prod-commands.sh
│   └── notify-on-completion.sh
└── commands/
    ├── review-ui-changes.md
    ├── generate-test-cases.md
    ├── investigate-bug-report.md
    └── draft-release-summary.md
```

```text
.claude/
└── skills/
    ├── api-review/
    │   ├── SKILL.md
    │   └── checklist.md
    ├── release-prep/
    │   ├── SKILL.md
    │   └── release-template.md
    └── docs-audit/
        ├── SKILL.md
        └── style-guide.md
```

```text
.claude/
└── skills/
    └── release-prep/
        ├── SKILL.md
        └── release-template.md
```

```text
.claude/
└── agents/
    ├── code-reviewer.md
    ├── security-auditor.md
    ├── docs-writer.md
    └── performance-checker.md
```

```text
.claude/
└── agents/
    └── code-reviewer.md
```

```text
.claude/
├── commands/
│   ├── review-pr.md
│   └── write-tests.md
├── skills/
│   ├── release-prep/
│   │   ├── SKILL.md
│   │   └── release-template.md
│   └── docs-audit/
│       ├── SKILL.md
│       └── style-guide.md
└── agents/
    ├── code-reviewer.md
    ├── security-auditor.md
    └── docs-writer.md
```

```text
.claude/
├── skills/
│   ├── release-prep/
│   │   ├── SKILL.md
│   │   └── release-template.md
│   ├── api-audit/
│   │   ├── SKILL.md
│   │   └── api-checklist.md
│   └── onboarding-docs/
│       ├── SKILL.md
│       └── docs-outline.md
└── agents/
    ├── code-reviewer.md
    ├── security-auditor.md
    └── documentation-editor.md
```

```text
your-project/
├── CLAUDE.md
├── CLAUDE.local.md
└── .claude/
    ├── settings.json
    ├── settings.local.json
    ├── rules/
    ├── hooks/
    ├── commands/
    ├── skills/
    └── agents/
~/.claude/
├── CLAUDE.md
├── settings.json
├── skills/
├── agents/
└── projects/
```

```text
your-project/
├── CLAUDE.md
└── .claude/
    ├── settings.json
    ├── rules/
    │   ├── api.md
    │   ├── testing.md
    │   └── security.md
    ├── hooks/
    │   ├── block-secrets-access.sh
    │   └── run-tests-before-stop.sh
    └── commands/
        ├── review-pr.md
        └── investigate-bug.md
```

```text
~/.claude/
├── CLAUDE.md
├── skills/
│   ├── article-outline/
│   │   ├── SKILL.md
│   │   └── outline-template.md
│   └── sql-review/
│       ├── SKILL.md
│       └── checklist.md
└── agents/
    ├── documentation-editor.md
    └── sql-reviewer.md
```

```text
your-project/
├── CLAUDE.md
├── CLAUDE.local.md
└── .claude/
    ├── settings.json
    ├── settings.local.json
    ├── rules/
    │   ├── backend.md
    │   ├── frontend.md
    │   ├── testing.md
    │   └── security.md
    ├── hooks/
    │   ├── block-dangerous-commands.sh
    │   ├── format-edits.sh
    │   └── run-checks-before-stop.sh
    ├── commands/
    │   ├── review-pr.md
    │   ├── write-tests.md
    │   └── summarize-changes.md
    ├── skills/
    │   └── release-prep/
    │       ├── SKILL.md
    │       └── release-template.md
    └── agents/
        ├── code-reviewer.md
        └── security-auditor.md
```

```text
your-project/
├── CLAUDE.md
└── .claude/
    ├── settings.json
    ├── rules/
    │   ├── testing.md
    │   └── security.md
    └── hooks/
        └── format-edits.sh
```

```text
.claude/
├── settings.json
├── rules/
│   ├── ui-components.md
│   ├── api-contracts.md
│   └── testing.md
├── hooks/
│   ├── lint-edits.sh
│   └── block-prod-commands.sh
├── commands/
│   ├── review-ui-changes.md
│   └── investigate-bug.md
└── agents/
    ├── ui-reviewer.md
    └── api-reviewer.md
```

```text
Common .claude/ mistakes
│
├── Overloaded CLAUDE.md
├── Too many folders too early
├── Shared files mixed with personal preferences
├── Weak or vague filenames
├── Old workflows left behind
└── Instructions stored in the wrong place
```

```text
Before diving into each component, it helps to look at the overall structure of an efficient .claude/ folder. Think of this as the target layout we want to build toward as the project becomes more sophisticated.
```

```text
This is where skills/ and agents/ come in.
```

